In [ ]:
!pip install \
    transformers==4.46.3 \
    peft==0.12.0 \
    accelerate==1.1.1 \
    trl==0.12.2 \
    bitsandbytes==0.44.1 \
    datasets==3.1.0 \
    sentencepiece \
    tiktoken \
    protobuf \
    safetensors \
    --quiet

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [2]:
!pip uninstall tensorflow keras -y --quiet

2026-11-19 08:56:01,698: 6071 WARNING autodynatrace - init: Could not initialize the OneAgent SDK, AgentState: 3


In [ ]:
import torch

print("=" * 50)
print("       FINE-TUNING SYSTEM CHECK")
print("=" * 50)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU memory      : {gpu_mem:.1f} GB")
print("=" * 50)

# Verify imports
try:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
    from datasets import Dataset
    print("✅ All imports successful!")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("   Restart kernel and re-run Cell 1")

In [3]:
import json

#train_file = "/home/dell/SLM_Engineering/scenario2_reference_3000_training.jsonl"
#train_file = "/home/dell/SLM_Engineering/scenario1_scenario2_min3_3000_training (1).jsonl"
#train_file = "/home/dell/SLM_Engineering/scenario2_exhaustive_remediation_fallthrough_3000_training (1).jsonl"
train_file ="/home/dell/SLM_Engineering/V1_Training_Data_Battery_Not_Charging_2000.jsonl"

# ============================================================
# Read JSONL file (one JSON object per line)
# ============================================================
training_data = []

with open(train_file, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if line:  # skip empty lines
            try:
                example = json.loads(line)
                training_data.append(example)
            except json.JSONDecodeError as e:
                print(f"⚠️  Line {line_num}: JSON parse error → {e}")

print("=" * 60)
print(f"✅ Successfully loaded {len(training_data)} training examples")
print("=" * 60)

# ============================================================
# Check the structure of the first example
# ============================================================
print("\n📋 Keys in first example:", list(training_data[0].keys()))
print()

# Detect format
first = training_data[0]
if "messages" in first:
    print("📌 Format: ChatML messages format (list of role/content dicts)")
    print(f"   Roles: {[m['role'] for m in first['messages']]}")
elif "system" in first and "user" in first and "assistant" in first:
    print("📌 Format: Flat format (system/user/assistant keys)")
    print("   Will convert to ChatML messages format for training")
else:
    print("⚠️  Unknown format — keys:", list(first.keys()))

✅ Successfully loaded 2000 training examples

📋 Keys in first example: ['messages']

📌 Format: ChatML messages format (list of role/content dicts)
   Roles: ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']


In [4]:
print("=" * 70)
print("       FIRST 5 TRAINING EXAMPLES — PREVIEW")
print("=" * 70)

for i, example in enumerate(training_data[:5]):
    print(f"\n{'─' * 70}")
    print(f"📝 EXAMPLE {i+1}")
    print(f"{'─' * 70}")

    # Handle both formats
    if "messages" in example:
        system_text = ""
        user_text = ""
        assistant_text = ""
        for msg in example["messages"]:
            if msg["role"] == "system":
                system_text = msg["content"]
            elif msg["role"] == "user":
                user_text = msg["content"]
            elif msg["role"] == "assistant":
                assistant_text = msg["content"]
    else:
        system_text = example.get("system", "")
        user_text = example.get("user", "")
        assistant_text = example.get("assistant", "")

    # Show system prompt (truncated — it's the same for all)
    print(f"\n🔧 SYSTEM: {system_text[:80]}...")
    print(f"   (length: {len(system_text)} chars)")

    # Show user input
    print(f"\n👤 USER:")
    # Extract device_id from user text
    for line in user_text.split("\n"):
        if "device_id" in line:
            print(f"   {line.strip()}")
        if "calibrated_confidence" in line:
            print(f"   {line.strip()}")
        if "evidence_tier" in line:
            print(f"   {line.strip()}")

    # Show first SHAP feature
    shap_lines = [l.strip() for l in user_text.split("\n") if l.strip().startswith("- ")]
    if shap_lines:
        print(f"   Top SHAP: {shap_lines[0]}")
        print(f"   ({len(shap_lines)} features total)")

    # Show assistant response summary
    print(f"\n🤖 ASSISTANT:")
    print(f"   (length: {len(assistant_text)} chars)")

    # Extract key outputs
    for line in assistant_text.split("\n"):
        line_stripped = line.strip()
        if line_stripped.startswith("L3:"):
            print(f"   ✅ {line_stripped}")
        if line_stripped.startswith("Confidence:"):
            print(f"   📊 {line_stripped}")
        if line_stripped.startswith("- Function:"):
            print(f"   💊 {line_stripped}")

print(f"\n{'=' * 70}")
print(f"📊 Total examples: {len(training_data)}")
print(f"{'=' * 70}")

       FIRST 5 TRAINING EXAMPLES — PREVIEW

──────────────────────────────────────────────────────────────────────
📝 EXAMPLE 1
──────────────────────────────────────────────────────────────────────

🔧 SYSTEM: You are the Reasoning & Execution engine for Battery Not Charging issues.

Your ...
   (length: 2163 chars)

👤 USER:

🤖 ASSISTANT:
   (length: 827 chars)

──────────────────────────────────────────────────────────────────────
📝 EXAMPLE 2
──────────────────────────────────────────────────────────────────────

🔧 SYSTEM: You are the Reasoning & Execution engine for Battery Not Charging issues.

Your ...
   (length: 2163 chars)

👤 USER:

🤖 ASSISTANT:
   (length: 980 chars)

──────────────────────────────────────────────────────────────────────
📝 EXAMPLE 3
──────────────────────────────────────────────────────────────────────

🔧 SYSTEM: You are the Reasoning & Execution engine for Battery Not Charging issues.

Your ...
   (length: 2163 chars)

👤 USER:

🤖 ASSISTANT:
   (length: 970 char

In [5]:
print("=" * 70)
print("       DATASET STATISTICS")
print("=" * 70)

# Collect stats
l3_counts = {}
confidence_counts = {}
remediation_counts = {}
system_lengths = []
user_lengths = []
assistant_lengths = []

for example in training_data:
    # Handle both formats
    if "messages" in example:
        system_text = ""
        user_text = ""
        assistant_text = ""
        for msg in example["messages"]:
            if msg["role"] == "system": system_text = msg["content"]
            elif msg["role"] == "user": user_text = msg["content"]
            elif msg["role"] == "assistant": assistant_text = msg["content"]
    else:
        system_text = example.get("system", "")
        user_text = example.get("user", "")
        assistant_text = example.get("assistant", "")

    system_lengths.append(len(system_text))
    user_lengths.append(len(user_text))
    assistant_lengths.append(len(assistant_text))

    # Extract L3
    for line in assistant_text.split("\n"):
        line_s = line.strip()
        if line_s.startswith("L3:") and "Primary" not in line_s and "Secondary" not in line_s:
            l3 = line_s.replace("L3:", "").strip()
            l3_counts[l3] = l3_counts.get(l3, 0) + 1
        elif line_s.startswith("L3 (Primary):"):
            l3 = line_s.replace("L3 (Primary):", "").strip()
            l3_counts[l3] = l3_counts.get(l3, 0) + 1
        if line_s.startswith("Confidence:"):
            conf = line_s.replace("Confidence:", "").strip()
            confidence_counts[conf] = confidence_counts.get(conf, 0) + 1
        if line_s.startswith("- Function:"):
            func = line_s.replace("- Function:", "").strip()
            remediation_counts[func] = remediation_counts.get(func, 0) + 1

# Print L3 distribution
print("\n📊 L3 Root Cause Distribution:")
print("-" * 40)
for l3, count in sorted(l3_counts.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f"  {l3[:40]:<42} {count:>2} {bar}")

# Print confidence distribution
print(f"\n📊 Confidence Distribution:")
print("-" * 40)
for conf, count in sorted(confidence_counts.items()):
    print(f"  {conf:<15} {count:>2}")

# Print remediation distribution
print(f"\n📊 Remediation Functions Used:")
print("-" * 40)
for func, count in sorted(remediation_counts.items(), key=lambda x: -x[1]):
    print(f"  {func:<35} {count:>2}")

# Print length stats
print(f"\n📏 Text Lengths (characters):")
print("-" * 40)
print(f"  System prompt: {min(system_lengths):,} - {max(system_lengths):,} (same for all: {'YES' if len(set(system_lengths))==1 else 'NO'})")
print(f"  User input:    {min(user_lengths):,} - {max(user_lengths):,} (avg: {sum(user_lengths)//len(user_lengths):,})")
print(f"  Assistant:     {min(assistant_lengths):,} - {max(assistant_lengths):,} (avg: {sum(assistant_lengths)//len(assistant_lengths):,})")

print(f"\n{'=' * 70}")
print(f"✅ Dataset is ready for fine-tuning!")
print(f"   Total examples: {len(training_data)}")
print(f"{'=' * 70}")

       DATASET STATISTICS

📊 L3 Root Cause Distribution:
----------------------------------------

📊 Confidence Distribution:
----------------------------------------

📊 Remediation Functions Used:
----------------------------------------

📏 Text Lengths (characters):
----------------------------------------
  System prompt: 2,163 - 2,163 (same for all: YES)
  User input:    68 - 108 (avg: 100)
  Assistant:     647 - 1,072 (avg: 893)

✅ Dataset is ready for fine-tuning!
   Total examples: 2000


In [6]:
import json
import os

# ============================================================
# PATHS — Update these if your files are in different locations
# ============================================================
model_dir = "/home/dell/SLM_Models/SmolLM2-360M-Instruct"
train_file = "/home/dell/SLM_Engineering/V1_Training_Data_Battery_Not_Charging_2000.jsonl"

# ============================================================
# Load JSONL (one JSON object per line)
# ============================================================
training_data = []

with open(train_file, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        line = line.strip()
        if line:
            try:
                example = json.loads(line)
                training_data.append(example)
            except json.JSONDecodeError as e:
                print(f"⚠️  Line {line_num}: Parse error → {e}")

print("=" * 60)
print(f"✅ Loaded {len(training_data)} training examples from:")
print(f"   {train_file}")
print("=" * 60)

# ============================================================
# Detect and show format
# ============================================================
first = training_data[0]
print(f"\n📋 Keys in each example: {list(first.keys())}")

if "messages" in first:
    roles = [m["role"] for m in first["messages"]]
    print(f"📌 Format: ChatML messages → {roles}")
    print("✅ Already in correct format — no conversion needed!")
elif "system" in first and "user" in first and "assistant" in first:
    print("📌 Format: Flat (system/user/assistant keys)")
    print("⚠️  Will need conversion to ChatML messages format")
else:
    print(f"⚠️  Unknown format — keys: {list(first.keys())}")

# Quick preview
print(f"\n📝 First example preview:")
if "messages" in first:
    for msg in first["messages"]:
        content_preview = msg["content"][:80].replace("\n", " ")
        print(f"   [{msg['role']:>9}] {content_preview}...")
else:
    print(f"   [system]    {first.get('system', '')[:80]}...")
    print(f"   [user]      {first.get('user', '')[:80]}...")
    print(f"   [assistant] {first.get('assistant', '')[:80]}...")

✅ Loaded 2000 training examples from:
   /home/dell/SLM_Engineering/V1_Training_Data_Battery_Not_Charging_2000.jsonl

📋 Keys in each example: ['messages']
📌 Format: ChatML messages → ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']
✅ Already in correct format — no conversion needed!

📝 First example preview:
   [   system] You are the Reasoning & Execution engine for Battery Not Charging issues.  Your ...
   [     user] shap_features: - PossibleValues: 0.950 - underpower_persist_seconds: 0.840 - bat...
   [assistant] THOUGHT 1: The highest SHAP feature is PossibleValues.  Looking up PossibleValue...
   [     user] [TOOL_RESPONSE] {"result": "Battery charge cap is disabled."}...
   [assistant] THOUGHT 2: Previous tool CheckBatteryChargeCap returned "Battery charge cap is d...
   [     user] [TOOL_RESPONSE] {"result": "Insufficient AC Adapter charge wattage detected. AC ...
   [assistant] THOUGHT 3: Previous tool GetACAdapterChargeWattageStat

In [7]:
# import os
# import json

# model_dir = "cycle2_2500_updated_format.jsonl"

# # Check if directory exists
# print(f"Directory exists: {os.path.exists(model_dir)}")
# print(f"\nFiles in directory:")
# if os.path.exists(model_dir):
#     for f in sorted(os.listdir(model_dir)):
#         size_mb = os.path.getsize(os.path.join(model_dir, f)) / (1024**2)
#         print(f"  {f:40s} {size_mb:.2f} MB")
    
#     # Check config.json
#     config_path = os.path.join(model_dir, "config.json")
#     if os.path.exists(config_path):
#         with open(config_path, "r") as f:
#             config = json.load(f)
#         print(f"\n📄 config.json contents:")
#         print(json.dumps(config, indent=2))
#         print(f"\n🔑 model_type = {config.get('model_type', '❌ MISSING!')}")
#     else:
#         print("\n❌ config.json NOT FOUND!")
# else:
#     print("❌ Directory does not exist! Check path.")

In [8]:
import os

path = "/home/dell/SLM_Engineering/V1_Training_Data_Battery_Not_Charging_2000.jsonl"

print(f"Path exists: {os.path.exists(path)}")

if os.path.isdir(path):
    print(f"\n📁 '{path}' is a directory. Files inside:")
    for f in sorted(os.listdir(path)):
        full_path = os.path.join(path, f)
        if os.path.isfile(full_path):
            size_mb = os.path.getsize(full_path) / (1024**2)
            print(f"  {f:40s} {size_mb:.2f} MB")

elif os.path.isfile(path):
    size_mb = os.path.getsize(path) / (1024**2)
    print(f"\n📄 '{path}' is a file.")
    print(f"   Size: {size_mb:.2f} MB")

else:
    print(f"\n❌ Path does not exist: {path}")

Path exists: True

📄 '/home/dell/SLM_Engineering/V1_Training_Data_Battery_Not_Charging_2000.jsonl' is a file.
   Size: 9.73 MB


In [9]:
import os
import json

file_path = "/home/dell/SLM_Engineering/V1_Training_Data_Battery_Not_Charging_2000.jsonl"

# Check if file exists
print(f"File exists: {os.path.exists(file_path)}")

if os.path.exists(file_path):
    if os.path.isfile(file_path):
        size_mb = os.path.getsize(file_path) / (1024**2)
        print(f"\n📄 File: {file_path}")
        print(f"   Size: {size_mb:.2f} MB")

        # Count lines / examples
        line_count = 0
        valid_json_count = 0
        invalid_lines = []

        with open(file_path, "r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                line_count += 1
                try:
                    json.loads(line)
                    valid_json_count += 1
                except json.JSONDecodeError as e:
                    invalid_lines.append((line_num, str(e)))

        print(f"\n📊 Line Summary:")
        print(f"   Total non-empty lines: {line_count}")
        print(f"   Valid JSON lines:      {valid_json_count}")
        print(f"   Invalid JSON lines:    {len(invalid_lines)}")

        if invalid_lines:
            print(f"\n⚠️ First few invalid lines:")
            for line_num, err in invalid_lines[:5]:
                print(f"   Line {line_num}: {err}")

        # Preview first example structure
        with open(file_path, "r", encoding="utf-8") as f:
            first_line = f.readline().strip()
            first_record = json.loads(first_line)
            print(f"\n📋 Keys in first record: {list(first_record.keys())}")
            if "messages" in first_record:
                roles = [m.get("role") for m in first_record["messages"]]
                print(f"   Roles in first record: {roles}")
    else:
        print(f"⚠️ {file_path} exists but is not a regular file (unexpected type).")
else:
    print(f"❌ File not found: {file_path}")

File exists: True

📄 File: /home/dell/SLM_Engineering/V1_Training_Data_Battery_Not_Charging_2000.jsonl
   Size: 9.73 MB

📊 Line Summary:
   Total non-empty lines: 2000
   Valid JSON lines:      2000
   Invalid JSON lines:    0

📋 Keys in first record: ['messages']
   Roles in first record: ['system', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant', 'user', 'assistant']


In [10]:
# import os
# import shutil

# model_dir = "SLM_Models/SmolLM2-135M-Instruct"

# # Mapping: current (wrong) name → correct name
# rename_map = {
#     "config (1).json":              "config.json",
#     "generation_config (1).json":   "generation_config.json",
#     "gitattributes (1)":            ".gitattributes",
#     "merges (1).txt":               "merges.txt",
#     "model (1).safetensors":        "model.safetensors",
#     "tokenizer (3).json":           "tokenizer.json",
#     "tokenizer_config (3).json":    "tokenizer_config.json",
#     "vocab (1).json":               "vocab.json",
#     "README (3).md":                "README.md",
# }

# print("🔧 Renaming files...\n")
# for old_name, new_name in rename_map.items():
#     old_path = os.path.join(model_dir, old_name)
#     new_path = os.path.join(model_dir, new_name)
#     if os.path.exists(old_path):
#         shutil.move(old_path, new_path)
#         print(f"  ✅ {old_name:40s} → {new_name}")
#     else:
#         print(f"  ⚠️  {old_name:40s}   (not found, skipping)")

# # Verify
# print(f"\n{'=' * 60}")
# print("📁 Files after renaming:")
# print(f"{'=' * 60}")
# for f in sorted(os.listdir(model_dir)):
#     size_mb = os.path.getsize(os.path.join(model_dir, f)) / (1024**2)
#     print(f"  {f:40s} {size_mb:.2f} MB")

# # Verify config.json now exists and has model_type
# import json
# config_path = os.path.join(model_dir, "config.json")
# if os.path.exists(config_path):
#     with open(config_path, "r") as f:
#         config = json.load(f)
#     print(f"\n🔑 model_type = {config.get('model_type', '❌ MISSING')}")
#     print("✅ Ready to load!")
# else:
#     print("\n❌ config.json still not found!")

In [11]:
import os
import json
from pathlib import Path
from transformers import AutoTokenizer, AutoConfig

model_dir = "/home/dell/SLM_Engineering/SLM_Models/SmolLM2-360M-Instruct"
model_path = Path(model_dir)

print("=" * 60)
print("MODEL CONFIGURATION DIAGNOSTICS")
print("=" * 60)

print("Working directory:", os.getcwd())
print("Model directory exists:", model_path.is_dir())

if model_path.is_dir():
    print("\nModel files:")
    for file in sorted(model_path.iterdir()):
        print(" ", file.name)

    print("\nRequired configuration files:")
    for filename in [
        "config.json",
        "tokenizer_config.json",
        "tokenizer.json",
        "special_tokens_map.json",
    ]:
        path = model_path / filename
        print(f"{filename}: {'FOUND' if path.is_file() else 'MISSING'}")

    config_path = model_path / "config.json"

    if config_path.is_file():
        with open(config_path, encoding="utf-8") as f:
            config = json.load(f)

        print("\nModel type:", config.get("model_type"))
        print("Architectures:", config.get("architectures"))

    print("\nAttempting AutoConfig...")
    try:
        model_config = AutoConfig.from_pretrained(
            model_dir,
            local_files_only=True
        )
        print("AutoConfig: SUCCESS")
    except Exception as e:
        print("AutoConfig: FAILED")
        print(type(e).__name__, str(e))

    print("\nAttempting AutoTokenizer...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_dir,
            local_files_only=True
        )
        print("AutoTokenizer: SUCCESS")
        print("Vocabulary size:", tokenizer.vocab_size)
    except Exception as e:
        print("AutoTokenizer: FAILED")
        print(type(e).__name__, str(e))

MODEL CONFIGURATION DIAGNOSTICS
Working directory: /home/dell/SLM_Engineering
Model directory exists: True

Model files:
  .ipynb_checkpoints
  README.md
  all_results.json
  config.json
  eval_results.json
  generation_config.json
  gitattributes
  merges.txt
  model.safetensors
  special_tokens_map.json
  tokenizer.json
  tokenizer_config.json
  train_results.json
  trainer_state.json
  training_args.bin
  vocab.json

Required configuration files:
config.json: FOUND
tokenizer_config.json: FOUND
tokenizer.json: FOUND
special_tokens_map.json: FOUND

Model type: llama
Architectures: ['LlamaForCausalLM']

Attempting AutoConfig...
AutoConfig: SUCCESS

Attempting AutoTokenizer...
AutoTokenizer: SUCCESS
Vocabulary size: 49152


In [12]:

from transformers import AutoTokenizer

model_dir = "/home/dell/SLM_Engineering/SLM_Models/SmolLM2-360M-Instruct"

# ============================================================
# Load tokenizer
# ============================================================
print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token
print(f"✅ Tokenizer loaded!")
print(f"   Vocab size: {tokenizer.vocab_size}")
print(f"   Pad token: {tokenizer.pad_token}")

# ============================================================
# Measure exact token length for EVERY example
# ============================================================
print(f"\n{'=' * 70}")
print(f"       TOKEN LENGTH ANALYSIS — ALL {len(training_data)} EXAMPLES")
print(f"{'=' * 70}")

token_lengths = []

for i, example in enumerate(training_data):
    # Handle both formats
    if "messages" in example:
        messages = example["messages"]
    else:
        # Convert flat format to messages
        messages = [
            {"role": "system", "content": example.get("system", "")},
            {"role": "user", "content": example.get("user", "")},
            {"role": "assistant", "content": example.get("assistant", "")}
        ]

    # Apply chat template (same as training will do)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Count tokens
    tokens = tokenizer(text, return_tensors="pt")
    num_tokens = tokens["input_ids"].shape[1]
    token_lengths.append(num_tokens)

    # Extract device_id for display
    user_content = messages[1]["content"] if len(messages) > 1 else ""
    device_id = "unknown"
    for line in user_content.split("\n"):
        if "device_id" in line:
            device_id = line.split(":")[-1].strip()
            break

    # Color-code by length
    if num_tokens > 2560:
        status = "🔴"
    elif num_tokens > 2048:
        status = "🟡"
    else:
        status = "🟢"

    bar = "█" * (num_tokens // 50)
    print(f"  {status} Example {i+1:>2} ({device_id:>8}): {num_tokens:>5} tokens {bar}")

# ============================================================
# Summary statistics
# ============================================================
print(f"\n{'=' * 70}")
print(f"📊 TOKEN LENGTH SUMMARY")
print(f"{'=' * 70}")
print(f"   Minimum:  {min(token_lengths):>5} tokens")
print(f"   Maximum:  {max(token_lengths):>5} tokens")
print(f"   Average:  {sum(token_lengths) // len(token_lengths):>5} tokens")
print(f"   Median:   {sorted(token_lengths)[len(token_lengths)//2]:>5} tokens")

# ============================================================
# Recommend max_length
# ============================================================
max_tokens = max(token_lengths)

if max_tokens <= 1536:
    recommended_max = 1536
elif max_tokens <= 2048:
    recommended_max = 2048
elif max_tokens <= 2560:
    recommended_max = 2560
elif max_tokens <= 3072:
    recommended_max = 3072
else:
    recommended_max = 4096

# Count how many fit at each threshold
for threshold in [1536, 2048, 2560, 3072, 4096]:
    fits = sum(1 for t in token_lengths if t <= threshold)
    pct = 100 * fits / len(token_lengths)
    marker = " ✅ RECOMMENDED" if threshold == recommended_max else ""
    print(f"   max_length={threshold}: {fits}/{len(token_lengths)} fit ({pct:.0f}%){marker}")

print(f"\n🎯 Recommended max_length = {recommended_max}")
print(f"   (covers all {len(token_lengths)} examples with no truncation)")

# Check for truncation risk
truncated = sum(1 for t in token_lengths if t > recommended_max)
if truncated > 0:
    print(f"\n⚠️  WARNING: {truncated} examples will be truncated!")
else:
    print(f"\n✅ ZERO examples will be truncated — perfect!")

⏳ Loading tokenizer...
✅ Tokenizer loaded!
   Vocab size: 49152
   Pad token: <|im_end|>

       TOKEN LENGTH ANALYSIS — ALL 2000 EXAMPLES
  🟢 Example  1 ( unknown):  1280 tokens █████████████████████████
  🟢 Example  2 ( unknown):  1425 tokens ████████████████████████████
  🟢 Example  3 ( unknown):  1410 tokens ████████████████████████████
  🟢 Example  4 ( unknown):  1474 tokens █████████████████████████████
  🟢 Example  5 ( unknown):  1107 tokens ██████████████████████
  🟢 Example  6 ( unknown):  1007 tokens ████████████████████
  🟢 Example  7 ( unknown):  1254 tokens █████████████████████████
  🟢 Example  8 ( unknown):  1424 tokens ████████████████████████████
  🟢 Example  9 ( unknown):  1024 tokens ████████████████████
  🟢 Example 10 ( unknown):  1405 tokens ████████████████████████████
  🟢 Example 11 ( unknown):  1124 tokens ██████████████████████
  🟢 Example 12 ( unknown):  1443 tokens ████████████████████████████
  🟢 Example 13 ( unknown):  1275 tokens █████████████████████████


In [13]:
from datasets import Dataset

# ============================================================
# UPDATE THIS if Cell 4 recommended a different value
# ============================================================
MAX_LENGTH = 1600  # ← Update based on Cell 4 output

print(f"📐 Using max_length = {MAX_LENGTH}")
print(f"⏳ Tokenizing {len(training_data)} examples...")

# ============================================================
# Format and tokenize each example
# ============================================================
def format_and_tokenize(example):
    # Handle both formats
    if "messages" in example:
        messages = example["messages"]
    else:
        messages = [
            {"role": "system", "content": example.get("system", "")},
            {"role": "user", "content": example.get("user", "")},
            {"role": "assistant", "content": example.get("assistant", "")}
        ]

    # Step 1: Apply chat template → formatted string
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Step 2: Tokenize with padding and truncation
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    # Step 3: Create labels (copy of input_ids)
    # The DataCollator will handle setting PAD → -100
    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

# Process all examples
tokenized_list = [format_and_tokenize(ex) for ex in training_data]
tokenized_dataset = Dataset.from_list(tokenized_list)

# ============================================================
# Verify tokenization
# ============================================================
print(f"\n{'=' * 60}")
print(f"✅ Dataset tokenized successfully!")
print(f"{'=' * 60}")
print(f"   Total examples:     {len(tokenized_dataset)}")
print(f"   Sequence length:    {len(tokenized_dataset[0]['input_ids'])} (max_length)")
print(f"   Columns:            {tokenized_dataset.column_names}")

# Show padding ratio for first 3 examples
print(f"\n📊 Padding analysis (first 5 examples):")
pad_token_id = tokenizer.pad_token_id
for i in range(min(5, len(tokenized_dataset))):
    ids = tokenized_dataset[i]["input_ids"]
    real_tokens = sum(1 for t in ids if t != pad_token_id)
    pad_tokens = sum(1 for t in ids if t == pad_token_id)
    pct_real = 100 * real_tokens / len(ids)
    print(f"   Example {i+1}: {real_tokens} real + {pad_tokens} padding = {len(ids)} total ({pct_real:.0f}% real)")

# Decode sample to verify it looks correct
print(f"\n🔍 Decoded sample (first 200 chars of Example 1):")
decoded = tokenizer.decode(tokenized_dataset[0]["input_ids"][:100], skip_special_tokens=False)
print(f"   {decoded[:200]}...")

📐 Using max_length = 1600
⏳ Tokenizing 2000 examples...

✅ Dataset tokenized successfully!
   Total examples:     2000
   Sequence length:    1600 (max_length)
   Columns:            ['input_ids', 'attention_mask', 'labels']

📊 Padding analysis (first 5 examples):
   Example 1: 1271 real + 329 padding = 1600 total (79% real)
   Example 2: 1414 real + 186 padding = 1600 total (88% real)
   Example 3: 1399 real + 201 padding = 1600 total (87% real)
   Example 4: 1463 real + 137 padding = 1600 total (91% real)
   Example 5: 1100 real + 500 padding = 1600 total (69% real)

🔍 Decoded sample (first 200 chars of Example 1):
   <|im_start|>system
You are the Reasoning & Execution engine for Battery Not Charging issues.

Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are conf...


In [14]:
# import torch
# from transformers import AutoModelForCausalLM
# from peft import LoraConfig, get_peft_model

# model_dir = "SLM_Models/SmolLM2-360M-Instruct"

# # ============================================================
# # STEP A: Load model in full precision (no quantization)
# # ============================================================
# print("⏳ Loading base model (full precision)...")

# model = AutoModelForCausalLM.from_pretrained(
#     model_dir,
#     torch_dtype=torch.float16,   # or torch.bfloat16 / torch.float32 depending on your GPU
#     device_map="auto",
#     local_files_only=True
# )

# print(f"✅ Base model loaded!")
# print(f"   GPU Memory: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")

# # ============================================================
# # STEP B: Apply LoRA adapters
# # ============================================================
# print(f"\n⏳ Applying LoRA adapters...")

# lora_config = LoraConfig(
#     r=16,                    # LoRA rank
#     lora_alpha=32,           # Scaling factor (2x rank)
#     target_modules=[
#         "q_proj", "k_proj", "v_proj", "o_proj",   # Attention layers
#         "gate_proj", "up_proj", "down_proj"         # MLP layers
#     ],
#     lora_dropout=0.05,       # Light regularization
#     bias="none",
#     task_type="CAUSAL_LM"
# )

# model = get_peft_model(model, lora_config)

# # ============================================================
# # Show trainable parameters
# # ============================================================
# trainable_params, total_params = model.get_nb_trainable_parameters()

# print(f"\n{'=' * 60}")
# print(f"✅ LoRA ADAPTERS APPLIED")
# print(f"{'=' * 60}")
# print(f"   Total parameters:     {total_params:>12,}")
# print(f"   Trainable parameters: {trainable_params:>12,}")
# print(f"   Trainable %:          {100 * trainable_params / total_params:>11.2f}%")
# print(f"   GPU Memory:           {torch.cuda.memory_allocated() / (1024**2):>11.0f} MB")
# print(f"{'=' * 60}")

In [15]:
# import torch
# from transformers import AutoModelForCausalLM, BitsAndBytesConfig
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# model_dir = "SLM_Models/SmolLM2-360M-Instruct"

# # ============================================================
# # STEP A: Load model in 4-bit quantization
# # ============================================================
# print("⏳ Loading model in 4-bit quantization (QLoRA)...")

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True
# )

# model = AutoModelForCausalLM.from_pretrained(
#     model_dir,
#     quantization_config=bnb_config,
#     device_map="auto",
#     local_files_only=True
# )

# model = prepare_model_for_kbit_training(model)

# print(f"✅ Base model loaded in 4-bit!")
# print(f"   GPU Memory: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")

# # ============================================================
# # STEP B: Apply LoRA adapters
# # ============================================================
# print(f"\n⏳ Applying LoRA adapters...")

# lora_config = LoraConfig(
#     r=16,                    # LoRA rank
#     lora_alpha=32,           # Scaling factor (2x rank)
#     target_modules=[
#         "q_proj", "k_proj", "v_proj", "o_proj",   # Attention layers
#         "gate_proj", "up_proj", "down_proj"         # MLP layers
#     ],
#     lora_dropout=0.05,       # Light regularization
#     bias="none",
#     task_type="CAUSAL_LM"
# )

# model = get_peft_model(model, lora_config)

# # ============================================================
# # Show trainable parameters
# # ============================================================
# trainable_params, total_params = model.get_nb_trainable_parameters()

# print(f"\n{'=' * 60}")
# print(f"✅ LoRA ADAPTERS APPLIED")
# print(f"{'=' * 60}")
# print(f"   Total parameters:     {total_params:>12,}")
# print(f"   Trainable parameters: {trainable_params:>12,}")
# print(f"   Trainable %:          {100 * trainable_params / total_params:>11.2f}%")
# print(f"   GPU Memory:           {torch.cuda.memory_allocated() / (1024**2):>11.0f} MB")
# print(f"{'=' * 60}")

In [16]:

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_dir = "SLM_Models/SmolLM2-360M-Instruct"

# ============================================================
# STEP A: Load model in 8-bit quantization
# ============================================================
print("⏳ Loading model in 8-bit quantization (INT8 LoRA)...")

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=0.0,
    llm_int8_enable_fp32_cpu_offload=False
)

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    quantization_config=bnb_config,
    device_map="auto",
    local_files_only=True
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False})

print("✅ Base model loaded in 8-bit!")
print(f"   GPU Memory: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")

# ============================================================
# STEP B: Apply LoRA adapters
# ============================================================
print("\n⏳ Applying LoRA adapters...")

lora_config = LoraConfig(
    r=16,                    # LoRA rank
    lora_alpha=32,           # Scaling factor (2x rank)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention layers
        "gate_proj", "up_proj", "down_proj"      # MLP layers
    ],
    lora_dropout=0.05,       # Light regularization
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# ============================================================
# Show trainable parameters
# ============================================================
trainable_params, total_params = model.get_nb_trainable_parameters()

print(f"\n{'=' * 60}")
print("✅ LoRA ADAPTERS APPLIED (8-BIT)")
print(f"{'=' * 60}")
print(f"   Total parameters:     {total_params:>12,}")
print(f"   Trainable parameters: {trainable_params:>12,}")
print(f"   Trainable %:          {100 * trainable_params / total_params:>11.2f}%")
print(f"   GPU Memory:           {torch.cuda.memory_allocated() / (1024**2):>11.0f} MB")
print(f"{'=' * 60}")

# ============================================================
# STEP C: Verify quantization
# ============================================================
print("\n🔍 Model quantization verification:")
# print(f"   Loaded in 8-bit: {getattr(model, "is_loaded_in_8bit", False)}")
print(f"   Loaded in 8-bit: {getattr(model, 'is_loaded_in_8bit', False)}")
print(f"   CUDA available: {torch.cuda.is_available()}")

model.print_trainable_parameters()

⏳ Loading model in 8-bit quantization (INT8 LoRA)...
✅ Base model loaded in 8-bit!
   GPU Memory: 481 MB

⏳ Applying LoRA adapters...

✅ LoRA ADAPTERS APPLIED (8-BIT)
   Total parameters:      370,504,640
   Trainable parameters:    8,683,520
   Trainable %:                 2.34%
   GPU Memory:                   514 MB

🔍 Model quantization verification:
   Loaded in 8-bit: True
   CUDA available: True
trainable params: 8,683,520 || all params: 370,504,640 || trainable%: 2.3437


In [17]:

# import torch
# from transformers import AutoModelForCausalLM
# from peft import LoraConfig, get_peft_model

# model_dir = "/home/dell/SLM_Engineering/SLM_Models/SmolLM2-360M-Instruct"

# # ============================================================
# # STEP A: Load model in 16-bit precision (FP16)
# # ============================================================
# print("⏳ Loading model in 16-bit precision (FP16)...")

# model = AutoModelForCausalLM.from_pretrained(
#     model_dir,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     local_files_only=True
# )

# print("✅ Base model loaded in FP16!")
# print(f"   GPU Memory: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")

# # ============================================================
# # STEP B: Apply LoRA adapters
# # ============================================================
# print("\n⏳ Applying LoRA adapters...")

# lora_config = LoraConfig(
#     r=16,                    # LoRA rank
#     lora_alpha=32,           # Scaling factor (2x rank)
#     target_modules=[
#         "q_proj", "k_proj", "v_proj", "o_proj",   # Attention layers
#         "gate_proj", "up_proj", "down_proj"      # MLP layers
#     ],
#     lora_dropout=0.05,       # Light regularization
#     bias="none",
#     task_type="CAUSAL_LM"
# )

# model = get_peft_model(model, lora_config)

# # ============================================================
# # Show trainable parameters
# # ============================================================
# trainable_params, total_params = model.get_nb_trainable_parameters()

# print(f"\n{'=' * 60}")
# print("✅ LoRA ADAPTERS APPLIED (FP16)")
# print(f"{'=' * 60}")
# print(f"   Total parameters:     {total_params:>12,}")
# print(f"   Trainable parameters: {trainable_params:>12,}")
# print(f"   Trainable %:          {100 * trainable_params / total_params:>11.2f}%")
# print(f"   GPU Memory:           {torch.cuda.memory_allocated() / (1024**2):>11.0f} MB")
# print(f"{'=' * 60}")

# # ============================================================
# # STEP C: Verify model precision
# # ============================================================
# print("\n🔍 Model precision verification:")
# print(f"   Base model dtype: {model.get_input_embeddings().weight.dtype}")
# print(f"   CUDA available:   {torch.cuda.is_available()}")

# # Display LoRA trainable parameters
# model.print_trainable_parameters()


In [18]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

output_dir = "/home/dell/SLM_Engineering/V1_Output_Battery_Not Charging"

# ============================================================
# Training configuration
# ============================================================
training_args = TrainingArguments(
    output_dir=output_dir,

    # --- Epochs & Batching ---
    num_train_epochs=3,
    per_device_train_batch_size=1,       # 1 example per forward pass
    gradient_accumulation_steps=4,        # Effective batch = 4

    # --- Learning Rate ---
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,

    # # --- Precision & Optimizer ---
    # fp16=True,
    # optim="paged_adamw_8bit",
    
    # --- Precision & Optimizer ---
    fp16=True,  # FP16 compute for INT8 base + LoRA; base weights remain INT8
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",

    # --- Regularization ---
    max_grad_norm=0.3,
    weight_decay=0.01,

    # --- Logging & Saving ---
    logging_steps=1,                      # Log EVERY step
    save_strategy="epoch",
    save_total_limit=3,                   # Keep last 3 checkpoints

    # --- Misc ---
    report_to="none",                     # No wandb/tensorboard
    remove_unused_columns=False
)

# ============================================================
# Data collator — handles PAD → -100 in labels
# ============================================================
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False    # Causal LM, not masked LM
)

# ============================================================
# Trainer
# ============================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# ============================================================
# Print training plan
# ============================================================
total_steps = len(tokenized_dataset) * training_args.num_train_epochs
steps_per_epoch = len(tokenized_dataset)
updates_per_epoch = steps_per_epoch // training_args.gradient_accumulation_steps

print("=" * 60)
print("       TRAINING PLAN")
print("=" * 60)
print(f"   Dataset size:          {len(tokenized_dataset)} examples")
print(f"   Epochs:                {training_args.num_train_epochs}")
print(f"   Batch size:            {training_args.per_device_train_batch_size}")
print(f"   Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"   Effective batch size:  {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Steps per epoch:       {steps_per_epoch}")
print(f"   Weight updates/epoch:  ~{updates_per_epoch}")
print(f"   Total steps:           {total_steps}")
print(f"   Total weight updates:  ~{updates_per_epoch * int(training_args.num_train_epochs)}")
print(f"   Learning rate:         {training_args.learning_rate}")
print(f"   Warmup steps:          {training_args.warmup_steps}")
print(f"   Output dir:            {output_dir}")
print("=" * 60)

# ============================================================
# START TRAINING
# ============================================================
print("\n🚀 STARTING FINE-TUNING...")
print("   Watch the loss — it should decrease over time\n")

train_result = trainer.train()

# ============================================================
# Training complete — show results
# ============================================================
print("\n" + "=" * 60)
print("       ✅ FINE-TUNING COMPLETE!")
print("=" * 60)
print(f"   Total training time: {train_result.metrics['train_runtime']:.0f} seconds")
print(f"   Final loss:          {train_result.metrics['train_loss']:.4f}")

print(f"   GPU Memory Peak:     {torch.cuda.max_memory_allocated() / (1024**2):.0f} MB")
print("=" * 60)

       TRAINING PLAN
   Dataset size:          2000 examples
   Epochs:                3
   Batch size:            1
   Gradient accumulation: 4
   Effective batch size:  4
   Steps per epoch:       2000
   Weight updates/epoch:  ~500
   Total steps:           6000
   Total weight updates:  ~1500
   Learning rate:         0.0002
   Warmup steps:          10
   Output dir:            /home/dell/SLM_Engineering/V1_Output_Battery_Not Charging

🚀 STARTING FINE-TUNING...
   Watch the loss — it should decrease over time



/opt/conda/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:316: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


A: torch.Size([1600, 960]), B: torch.Size([960, 960]), C: (1600, 960); (lda, ldb, ldc): (c_int(51200), c_int(30720), c_int(51200)); (m, n, k): (c_int(1600), c_int(960), c_int(960))


Exception: cublasLt ran into an error!

In [21]:
import os

# ============================================================
# Save LoRA adapters + tokenizer
# ============================================================
adapter_dir = "/home/dell/SLM_Engineering/Final_FP16_3000_final_adapter_smollm"   #FP16 bit adapter

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# ============================================================
# Verify saved files
# ============================================================
print("=" * 60)
print("       ✅ FINE-TUNED ADAPTERS SAVED")
print("=" * 60)
print(f"   Directory: {adapter_dir}")
print()

total_size = 0
for f in sorted(os.listdir(adapter_dir)):
    path = os.path.join(adapter_dir, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        total_size += size_mb
        print(f"   ✅ {f}: {size_mb:.2f} MB")

print(f"\n   📦 Total adapter size: {total_size:.2f} MB")
print(f"   (vs base model ~1,000 MB — LoRA is much smaller!)")
print("=" * 60)
print()
print("📋 What was saved:")
print("   - adapter_model.safetensors → LoRA weight deltas")
print("   - adapter_config.json → LoRA configuration (r=16, alpha=32)")
print("   - tokenizer files → needed for inference")
print()
print("📋 To use this model later:")
print(f"   1. Load base model from: SLM_Engineering")
print(f"   2. Load LoRA adapter from: {adapter_dir}")
print(f"   3. Merge and generate!")

       ✅ FINE-TUNED ADAPTERS SAVED
   Directory: /home/dell/SLM_Engineering/Final_FP16_3000_final_adapter_smollm

   ✅ README.md: 0.00 MB
   ✅ adapter_config.json: 0.00 MB
   ✅ adapter_model.safetensors: 33.18 MB
   ✅ merges.txt: 0.44 MB
   ✅ special_tokens_map.json: 0.00 MB
   ✅ tokenizer.json: 3.36 MB
   ✅ tokenizer_config.json: 0.00 MB
   ✅ vocab.json: 0.76 MB

   📦 Total adapter size: 37.76 MB
   (vs base model ~1,000 MB — LoRA is much smaller!)

📋 What was saved:
   - adapter_model.safetensors → LoRA weight deltas
   - adapter_config.json → LoRA configuration (r=16, alpha=32)
   - tokenizer files → needed for inference

📋 To use this model later:
   1. Load base model from: SLM_Engineering
   2. Load LoRA adapter from: /home/dell/SLM_Engineering/Final_FP16_3000_final_adapter_smollm
   3. Merge and generate!
